In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import linregress
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Read in data
data = pd.read_csv("~/Documents/2023:2024/Data/Exported data/superager.csv")
print(f"Number of participants: {len(data)}")

In [ ]:
def process_data(data, n_timepoints, mem_vars):
    """Process the dataset to calculate memory composites and slopes of change.

    Args:
        data (pd.DataFrame): The dataset containing memory variables.
        n_timepoints (int): Number of timepoints in the dataset.
        mem_vars (list): List of memory variable names to process.  
    """
    data = data.copy()
    data.reset_index(drop=True, inplace=True)

    # Scale memory variables
    for i in range(1, n_timepoints + 1):
        for mem_var in mem_vars:
            base_col_name = f"w1_{mem_var}"
            col_name = f"w{i}_{mem_var}"

            if base_col_name in data.columns and col_name in data.columns:
                baseline_values = data[base_col_name]

                baseline_mean = baseline_values.mean(skipna=True)
                baseline_sd = baseline_values.std(skipna=True)

                if baseline_sd != 0:  
                    scaled_values = (data[col_name] - baseline_mean) / baseline_sd
                    data[f"sc_{col_name}"] = scaled_values.astype(float)
                else:
                    print(f"Standard deviation is zero for {base_col_name}, skipping scaling.")
            else:
                print(f"Column not found: {col_name} or {base_col_name}")

    # Calculate memory composites
    for i in range(1, n_timepoints + 1):
        mem_cols = [f"sc_w{i}_{mem_var}" for mem_var in mem_vars]
        valid_columns = [col for col in mem_cols if col in data.columns]
        if valid_columns:
            data[f"w{i}_memory"] = data[valid_columns].mean(axis=1, skipna=True)

    # Calculate slopes
    for i in range(len(data)):
        em_data = data.loc[
            i, [f"w{j}_memory" for j in range(1, n_timepoints + 1) if f"w{j}_memory" in data.columns]
        ].astype(float)
        age = data.loc[i, [f"w{j}_age" for j in range(1, n_timepoints + 1) if f"w{j}_age" in data.columns]].astype(
            float
        )

        if len(age) > 1 and 'w2_memory' in em_data.index and not em_data.isna().all(): # Only calculate this for participants with follow-up data
            em_slope, _, _, _, _ = linregress(age, em_data)
            data.at[i, "memory_slopes"] = em_slope
            data.at[i, "mem_time"] = age.max() - age.min()
        else:
            data.at[i, "memory_slopes"] = np.nan
            data.at[i, "mem_time"] = np.nan

    return data

In [ ]:
# Subset the data to include only BBHI participants
bbhi_df = data[data["id"] > 10000]

In [ ]:
# NOTE that for BBHI it really needs to be the Face-Name task because otherwise the practice effects are overwhelming. 
# However, the RAVLT variables are still in the df for BBHI so the code can easily be changed to include the RAVLT

# Subset the data to include only BBHI participants
bbhi_df = data[data["id"] > 10000]

# Define memory variables
# memory_vars = ["crn30", "cro30", "crn", "cro"]
memory_vars = ["ravlt_total", "delayed_recall_raw"]

# Specify the number of timepoints
n_timepoints = 2

# Apply the function
bbhi_df = process_data(bbhi_df, n_timepoints, memory_vars)

# bbhi_df[["id", "memory_slopes", "mem_time", "sc_w1_crn30", "sc_w2_crn30"]].head()
bbhi_df[["id", "memory_slopes", "mem_time", "sc_w1_ravlt_total", "sc_w2_ravlt_total"]].head()

In [ ]:
# Apply the function to BBHI senior data
bbhi_senior_df = data[data["id"] < 10000]

# Define memory variables
memory_vars = ["ravlt_total", "delayed_recall_raw"]

# Specify the number of timepoints
n_timepoints = 2

# Apply the function
bbhi_senior_df = process_data(bbhi_senior_df, n_timepoints, memory_vars)

# If mem_time = 0 replace with NaN
bbhi_senior_df.loc[bbhi_senior_df["mem_time"] == 0, "mem_time"] = np.nan

bbhi_senior_df[["id", "memory_slopes", "mem_time", "sc_w1_ravlt_total", "sc_w2_ravlt_total"]].head()

In [ ]:
# Merge the dfs back together
common_columns = [col for col in bbhi_df.columns if col in bbhi_senior_df.columns]

# Merge on all common columns
data = pd.merge(bbhi_df, bbhi_senior_df, on=common_columns, how="outer")

data.sample(5)

This defines maintainer as a participant with an above average baseline memory score and memory slope, as has been done in [Josefsson et al., 2012](https://www.asc.ohio-state.edu/statistics/statgen/joul_aut2012/jgs.pdf).

In [ ]:
# Filter columns and convert to long format
df = data[["w1_age", "w2_age", "w1_memory", "w2_memory", "w1_delayed_recall_raw", "w2_delayed_recall_raw", "id", "superager"]]

df_long = pd.wide_to_long(
    df.rename(columns={
        "w1_age": "age_w1",
        "w2_age": "age_w2",
        "w1_memory": "memory_w1",
        "w2_memory": "memory_w2",
        "w1_delayed_recall_raw": "ravlt_delayed_w1",
        "w2_delayed_recall_raw": "ravlt_delayed_w2"
    }),
    stubnames=["age", "memory", "ravlt_delayed"],
    i="id", 
    j="wave",
    sep="_w"
).reset_index()

# Plot long data memory vs age
plt.figure(figsize=(8,6))
sns.lineplot(data=df_long, x="age", y="memory",
             units="id",  # connect points by id
             estimator=None,
             color="green", marker="o", alpha=0.6)
sns.regplot(data=df_long, x="age", y="memory", scatter=False, color="black", ci=None)
plt.xlabel("Age")
plt.ylabel("Memory")
plt.show()

# Plot long data ravlt_delayed vs age
df_long = df_long.copy()
df_long["sa_group"] = df_long["superager"].map({1: "Superager", 0: "Other"})

plt.figure(figsize=(8,6))
sns.lineplot(
    data=df_long, x="age", y="ravlt_delayed",
    hue="sa_group", units="id", estimator=None,
    marker="o", alpha=0.6, linewidth=1,
    palette={"Superager": "green", "Other": "gray"}
)
sns.regplot(data=df_long, x="age", y="ravlt_delayed", scatter=False, color="black", ci=None)
plt.xlabel("Age"); plt.ylabel("RAVLT Delayed Recall"); plt.legend(title="")
plt.show()

# Plot memory slopes vs age
plt.figure(figsize=(8,6))
sns.regplot(data=data, x="w1_age", y="memory_slopes", scatter_kws={"alpha":0.6}, ci=None, color="black")
plt.xlabel("Baseline Age")
plt.ylabel("Memory Change")
plt.axhline(0, color="blue", linestyle="--", linewidth=1)  
plt.show()

# Model controlling for sex and edu
model_slopes = smf.ols("memory_slopes ~ w1_age + sex + YoE", data=data).fit()
print(model_slopes.summary())

In [ ]:
# Generate a new variable 'maintainer' for participants who are >=0 on memory slopes
data["maintainer"] = np.where(
    data["memory_slopes"].isna(), np.nan, # Check if memory_slopes is NA and if NA, assign np.nan
    np.where(data["memory_slopes"] >= 0, 1, 0)
)

# Count how many particpants have NA for memory slopes
na_count = data["memory_slopes"].isna().sum()

# Summarize the data
row_count = len(data)
maintainer_count = data["maintainer"].sum()
decliner_count = row_count - maintainer_count - na_count

print(f"Number of participants: {row_count}")
print(f"Number of maintainers: {maintainer_count:.0f}")
print(f"Number of decliners: {decliner_count:.0f}")

# Get basic info about maintainers
maintainer_df = data[data["maintainer"] == 1]
average_age = maintainer_df["w1_age"].mean()
standard_deviation = maintainer_df["w1_age"].std()

print(f"Average maintainer age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

# Get basic info about controls
decliner_df = data[data["maintainer"] == 0]
average_age = decliner_df["w1_age"].mean()
standard_deviation = decliner_df["w1_age"].std()

print(f"Average decliner age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

In [ ]:
# Filter the df to include only the needed variables
clean_df = data[
    [
        "id",
        "w1_age",
        "w2_age",
        "YoE",
        "sex",
        "superager",
        "w1_memory",
        "w2_memory",
        "memory_slopes",
        "mem_time",
        "maintainer",
    ]
]

# Export this df to a csv to use for future analysis
clean_df.to_csv("/Users/rachelmorse/Documents/2023:2024/Data/Exported data/maintainer_superager_data.csv", index=False)

clean_df.sample(5)